# Kaapi Bricks — MAS Supervisor Evaluation Dataset

Creates an MLflow-managed evaluation dataset in Unity Catalog with 22 routing examples for the MAS Supervisor Agent.

**Table:** `fevm_cme_conde_catalog.kaapi_bricks.supervisor_examples`

**Agents:**
- `knowledge_assistant` — recipes, policies, training, maintenance, suppliers
- `analytics` — sales, revenue, inventory, customers, promotions
- `operations_advisor` — weather, calendar, planning, staffing, forecasting

In [ ]:
import mlflow
import mlflow.genai.datasets
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.remote(serverless=True).getOrCreate()

UC_TABLE = "fevm_cme_conde_catalog.kaapi_bricks.supervisor_examples"
print(f"Creating dataset: {UC_TABLE}")

In [ ]:
# Create the MLflow-managed dataset
eval_dataset = mlflow.genai.datasets.create_dataset(
    uc_table_name=UC_TABLE
)
print(f"Dataset created: {eval_dataset.uc_table_name}")

In [ ]:
# Define 22 routing examples
records = [
    # ===== Knowledge Assistant (6 examples) =====
    {
        "inputs": {"query": "How do I make a Classic Filter Coffee with the tumbler-davara technique?"},
        "expectations": {"guidelines": ["Should be routed to knowledge_assistant. This is a recipe and drink preparation question. The answer comes from the barista training manual and drink recipes SOP documents."]}
    },
    {
        "inputs": {"query": "What is our food safety policy for milk handling?"},
        "expectations": {"guidelines": ["Should be routed to knowledge_assistant. FSSAI compliance and food safety policies are documented in the food safety policy PDF."]}
    },
    {
        "inputs": {"query": "What are the franchise requirements for opening a new Kaapi Bricks location?"},
        "expectations": {"guidelines": ["Should be routed to knowledge_assistant. Franchise operations, financial requirements, and setup procedures are in the franchise operations guide."]}
    },
    {
        "inputs": {"query": "How do I maintain the brass filter coffee maker?"},
        "expectations": {"guidelines": ["Should be routed to knowledge_assistant. Equipment maintenance procedures including cleaning schedules and troubleshooting are in the equipment maintenance guide."]}
    },
    {
        "inputs": {"query": "What are the ingredients and recipe for Bella Kaapi Jaggery coffee?"},
        "expectations": {"guidelines": ["Should be routed to knowledge_assistant. Drink recipes with ingredients, prices, and preparation steps are in the drink recipes SOP document."]}
    },
    {
        "inputs": {"query": "What supplier provides our Coorg Arabica beans and what are their payment terms?"},
        "expectations": {"guidelines": ["Should be routed to knowledge_assistant. Supplier details, contracts, lead times, and payment terms are in the supplier agreements summary document."]}
    },

    # ===== Analytics / Genie (8 examples) =====
    {
        "inputs": {"query": "What are the top 5 selling drinks this month?"},
        "expectations": {"guidelines": ["Should be routed to analytics. Requires SQL query joining orders, order_items, and products tables to rank products by quantity sold."]}
    },
    {
        "inputs": {"query": "Show me revenue for the Koramangala store last week"},
        "expectations": {"guidelines": ["Should be routed to analytics. Revenue data comes from summing order_total in the orders table filtered by store and date range."]}
    },
    {
        "inputs": {"query": "Compare sales between Koramangala and Indiranagar stores"},
        "expectations": {"guidelines": ["Should be routed to analytics. Store comparison requires SQL queries on orders table grouped by store_id with joins to stores table."]}
    },
    {
        "inputs": {"query": "Which ingredients are running low in inventory?"},
        "expectations": {"guidelines": ["Should be routed to analytics. Inventory levels come from summing quantity in inventory_transactions table grouped by ingredient_id and store_id."]}
    },
    {
        "inputs": {"query": "How many customers are in our Gold loyalty tier?"},
        "expectations": {"guidelines": ["Should be routed to analytics. Customer loyalty tier data is in the customers table, filtered by loyalty_tier column."]}
    },
    {
        "inputs": {"query": "How did the Morning Rush Hour promotion perform?"},
        "expectations": {"guidelines": ["Should be routed to analytics. Promotion performance requires SQL joining promotion_redemptions with orders and promotions tables."]}
    },
    {
        "inputs": {"query": "What is the average order value across all stores?"},
        "expectations": {"guidelines": ["Should be routed to analytics. Requires AVG(order_total) aggregation on the orders table."]}
    },
    {
        "inputs": {"query": "Show me daily order count for the last 30 days"},
        "expectations": {"guidelines": ["Should be routed to analytics. Time series query counting orders grouped by order_date for the last 30 days."]}
    },

    # ===== Operations Advisor / MCP (8 examples) =====
    {
        "inputs": {"query": "What should I prepare for tomorrow at the Koramangala store?"},
        "expectations": {"guidelines": ["Should be routed to operations_advisor. This is an operational planning question that needs live weather forecast, Indian holiday/festival calendar check, and preparation recommendations."]}
    },
    {
        "inputs": {"query": "How should I staff for this weekend?"},
        "expectations": {"guidelines": ["Should be routed to operations_advisor. Staffing planning needs calendar awareness (weekend detection) and demand forecasting based on day-of-week patterns."]}
    },
    {
        "inputs": {"query": "Will the weather affect our sales tomorrow?"},
        "expectations": {"guidelines": ["Should be routed to operations_advisor. Weather impact analysis requires the live weather forecast tool to check temperature, rain, and humidity."]}
    },
    {
        "inputs": {"query": "Is there any festival coming up that I should prepare for?"},
        "expectations": {"guidelines": ["Should be routed to operations_advisor. Festival and holiday awareness comes from the Indian calendar tool which checks Diwali, Holi, Pongal, Eid, and other festivals."]}
    },
    {
        "inputs": {"query": "What demand should I expect for Friday evening?"},
        "expectations": {"guidelines": ["Should be routed to operations_advisor. Friday evening demand forecasting is an operational planning question that needs day-of-week awareness and historical pattern analysis."]}
    },
    {
        "inputs": {"query": "Should I order extra milk for next week?"},
        "expectations": {"guidelines": ["Should be routed to operations_advisor. Forward-looking supply planning needs weather forecast and calendar context to estimate upcoming demand."]}
    },
    {
        "inputs": {"query": "How will Diwali affect our stores?"},
        "expectations": {"guidelines": ["Should be routed to operations_advisor. Festival impact planning requires calendar awareness to understand Diwali dates and historical festival demand patterns."]}
    },
    {
        "inputs": {"query": "What cold drinks should I push if it is hot tomorrow?"},
        "expectations": {"guidelines": ["Should be routed to operations_advisor. Weather-based product recommendations need the live weather forecast to determine temperature and suggest appropriate drink categories."]}
    },
]

print(f"Prepared {len(records)} examples")
print(f"  knowledge_assistant: {sum(1 for r in records if 'knowledge_assistant' in r['expectations']['guidelines'][0])}")
print(f"  analytics: {sum(1 for r in records if 'analytics' in r['expectations']['guidelines'][0])}")
print(f"  operations_advisor: {sum(1 for r in records if 'operations_advisor' in r['expectations']['guidelines'][0])}")

In [ ]:
# Add records to the dataset
eval_dataset.merge_records(records)
print(f"Successfully added {len(records)} examples to {UC_TABLE}")

In [ ]:
# Verify — read back from the dataset
df = spark.table(UC_TABLE)
print(f"Total rows: {df.count()}")
df.select("eval_id", "request", "guidelines").show(5, truncate=60)

## How to use this dataset

### In MAS Supervisor (Agent Bricks UI)
Import this table as the evaluation dataset for the Kaapi Bricks HQ supervisor.

### In MLflow Evaluation
```python
existing = mlflow.genai.datasets.get_dataset(
    "fevm_cme_conde_catalog.kaapi_bricks.supervisor_examples"
)

results = mlflow.genai.evaluate(
    data=existing,
    predict_fn=my_supervisor_fn,
    scorers=[ExpectationsGuidelines()]
)
```